<a href="https://colab.research.google.com/github/dhiyasalmas/Machine_Learning/blob/main/Prediksi_hujan.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Impor library dan persiapan

In [ ]:
from pandas import DataFrame, Series
import pandas as pd             # Analisis data dalam dataframe
import matplotlib.pyplot as plt # Plot data standar
import numpy as np              # Analisis numerik
import seaborn as sns           # Visualisasi statistik
from datetime import datetime   # Pengolahan data waktu/tanggal

# untuk fungsi machine learning standar
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# untuk xgboost
from xgboost import XGBRegressor

# untuk neural network
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.utils import to_categorical


In [ ]:
def persiapan(folder_data):
    df_climate_data_staging = pd.read_csv(f'{folder_data}/climate_data.csv')
    df_climate_data_staging.head()

    kolom_pengganti = {
        'Tn': 'suhu_min',
        'Tx': 'suhu_maks',
        'Tavg': 'suhu_rata2',
        'RH_avg': 'kelembaban_rata2',
        'RR': 'curah_hujan',
        'ss': 'durasi_sinar_matahari_jam',
        'ff_x': 'kecepatan_angin_maks',
        'ddd_x': 'arah_angin_kecepatan_maks',
        'ff_avg': 'kecepatan_angin_rata2',
        'ddd_car': 'arah_angin_tersering',
        'station_id': 'id_stasiun',
        'date': 'tanggal_pencatatan'
    }

    df_climate_data = df_climate_data_staging.rename(columns=kolom_pengganti)
    df_provinces = pd.read_csv(f'{folder_data}/province_detail.csv')
    df_stations = pd.read_csv(f'{folder_data}/station_detail.csv')

    '''
    Pengaturan - Transformasi
    '''

    # mengubah format datetime menjadi format US
    df_climate_data = df_climate_data.assign(tanggal_pencatatan=df_climate_data['tanggal_pencatatan'].apply(lambda x: datetime.strptime(x, '%d-%m-%Y')))

    # mengonversi indeks ke tipe DatetimeIndex
    df_climate_data.set_index(pd.DatetimeIndex(df_climate_data.index))

    # menghasilkan set denormalisasi dengan menggabungkan semua dataframe
    df_climate_data = df_climate_data\
        .merge(df_stations, how='inner', left_on='id_stasiun', right_on='station_id')\
            .merge(df_provinces, how='inner', left_on='province_id', right_on='province_id')


    '''
    Memfilter data ekstrem dari grafik:
    '''
    df_climate_data = df_climate_data[(df_climate_data.suhu_min > 0) & (df_climate_data.suhu_maks < 50)]
    df_climate_data = df_climate_data[(df_climate_data.suhu_rata2 > 0) & (df_climate_data.suhu_rata2 < 50)]
    df_climate_data = df_climate_data[(df_climate_data.kelembaban_rata2 <= 100)]
    df_climate_data = df_climate_data[(df_climate_data.durasi_sinar_matahari_jam <= 20)]
    df_climate_data = df_climate_data[(df_climate_data.suhu_maks > df_climate_data.suhu_min)]
    df_climate_data = df_climate_data[(df_climate_data.curah_hujan >= 0) & (df_climate_data.curah_hujan < 500)]

    # Extract features: latitude, longitude, and month
    df_climate_data['bulan'] = pd.to_datetime(df_climate_data['tanggal_pencatatan']).dt.month
    # Remove rows with missing values (or handle them in other ways)
    df_climate_data = df_climate_data.dropna()

    # Create the rain_category based on curah_hujan (rainfall)
    df_climate_data['kategori_hujan'] = pd.cut(
        df_climate_data['curah_hujan'],
        bins=[-np.inf, 0, 5.0, np.inf],
        labels=[0, 1, 2]
    )


    return df_climate_data

In [ ]:
folder_data = "https://raw.githubusercontent.com/rizkiokt/dashboard-cuaca/refs/heads/main/data"
df_cuaca = persiapan(folder_data)

In [ ]:
df_cuaca.columns

Index(['tanggal_pencatatan', 'suhu_min', 'suhu_maks', 'suhu_rata2',
       'kelembaban_rata2', 'curah_hujan', 'durasi_sinar_matahari_jam',
       'kecepatan_angin_maks', 'arah_angin_kecepatan_maks',
       'kecepatan_angin_rata2', 'arah_angin_tersering', 'id_stasiun',
       'station_id', 'station_name', 'region_name', 'latitude', 'longitude',
       'region_id', 'province_id', 'province_name', 'bulan', 'kategori_hujan'],
      dtype='object')

In [ ]:
df_cuaca.head()

,tanggal_pencatatan,suhu_min,suhu_maks,suhu_rata2,kelembaban_rata2,curah_hujan,durasi_sinar_matahari_jam,kecepatan_angin_maks,arah_angin_kecepatan_maks,kecepatan_angin_rata2,...,station_id,station_name,region_name,latitude,longitude,region_id,province_id,province_name,bulan,kategori_hujan
0,2010-01-01,21.4,30.2,27.1,82.0,9.0,0.5,7.0,90.0,5.0,...,96001,Stasiun Meteorologi Maimun Saleh,Kota Sabang,5.87655,95.33785,20,1,Nanggroe Aceh Darussalam,1,2
1,2010-01-02,21.0,29.6,25.7,95.0,24.0,0.2,6.0,90.0,4.0,...,96001,Stasiun Meteorologi Maimun Saleh,Kota Sabang,5.87655,95.33785,20,1,Nanggroe Aceh Darussalam,1,2
2,2010-01-03,20.2,26.8,24.5,98.0,63.0,0.0,5.0,90.0,4.0,...,96001,Stasiun Meteorologi Maimun Saleh,Kota Sabang,5.87655,95.33785,20,1,Nanggroe Aceh Darussalam,1,2
3,2010-01-04,21.0,29.2,25.8,90.0,0.0,0.1,4.0,225.0,3.0,...,96001,Stasiun Meteorologi Maimun Saleh,Kota Sabang,5.87655,95.33785,20,1,Nanggroe Aceh Darussalam,1,0
7,2010-01-08,21.8,29.8,26.8,91.0,3.0,0.6,5.0,90.0,4.0,...,96001,Stasiun Meteorologi Maimun Saleh,Kota Sabang,5.87655,95.33785,20,1,Nanggroe Aceh Darussalam,1,1


In [ ]:
# Map numeric labels back to string labels for display
kode_kategori_hujan = {
    0: 'Tidak Hujan',
    1: 'Hujan Ringan',
    2: 'Hujan Lebat'
}

# Regresi dengan XGBoost

In [ ]:
X = df_cuaca[['latitude', 'longitude', 'bulan']]
y = df_cuaca['suhu_rata2']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)


In [ ]:
len(y_train)

297332

In [ ]:
len(y_test)

74333

In [ ]:
# Membuat model XGBoost
model = XGBRegressor(n_estimators = 100)

# Melatih model
model.fit(X_train, y_train)

# Melakukan prediksi pada data uji
y_pred = model.predict(X_test)

# Evaluasi model
mse = mean_squared_error(y_test, y_pred)
print(f"Mean Squared Error: {mse}")

# Membuat persentase error
error_percentage = (mse / np.mean(y_test)) * 100
print(f"Persentase Error: {error_percentage}%")

Mean Squared Error: 0.8776572073003907
Persentase Error: 3.272343676866527%


In [ ]:
# Contoh prediksi
# Jakarta = 6.1944° S, 106.8229° E
contoh_input = pd.DataFrame({'latitude': [6.1944], 'longitude': [106.8229], 'bulan': [7]})
prediksi_suhu = model.predict(contoh_input)
print(f"Prediksi Suhu Rata-rata: {prediksi_suhu[0]}")

Prediksi Suhu Rata-rata: 27.477535247802734


# Regresi dengan Neural Network

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# membuat model neural network
model = Sequential()
model.add(Dense(64, activation='relu', input_dim=X_train_scaled.shape[1]))
model.add(Dense(32, activation='relu'))
model.add(Dense(1, activation='linear')) # jumlah nodes di output layer mengikuti jumlah variabel

model.compile(optimizer='adam', loss='mean_squared_error')
model.summary()

/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense (Dense)                        │ (None, 64)                  │             256 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 32)                  │           2,080 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 1)                   │              33 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 2,369 (9.25 KB)

 Trainable params: 2,369 (9.25 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history = model.fit(X_train_scaled, y_train, epochs=25, batch_size=128, validation_split=0.2)

Epoch 1/25
1859/1859 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step - loss: 144.2753 - val_loss: 3.4216
Epoch 2/25
1859/1859 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - loss: 3.3022 - val_loss: 3.2821
Epoch 3/25
1859/1859 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - loss: 3.2277 - val_loss: 3.2415
Epoch 4/25
1859/1859 ━━━━━━━━━━━━━━━━━━━━ 7s 3ms/step - loss: 3.2018 - val_loss: 3.2607
Epoch 5/25
1859/1859 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - loss: 3.1622 - val_loss: 3.1714
Epoch 6/25
1859/1859 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - loss: 3.1400 - val_loss: 3.1478
Epoch 7/25
1859/1859 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step - loss: 3.1334 - val_loss: 3.0970
Epoch 8/25
1859/1859 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - loss: 3.0745 - val_loss: 3.0830
Epoch 9/25
1859/1859 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - loss: 3.0148 - val_loss: 3.0102
Epoch 10/25
1859/1859 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - loss: 2.9467 - val_loss: 2.9912
Epoch 11/25
1859/1859 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - loss: 2.9284 - val_loss: 2.8995
Epoch 12/25
1859/1859 ━━━━━━

In [ ]:
y_pred = model.predict(X_test_scaled)
mse = mean_squared_error(y_test, y_pred)
print(f"Mean Squared Error: {mse}")

# Membuat persentase error
error_percentage = (mse / np.mean(y_test)) * 100
print(f"Persentase Error: {error_percentage}%")

2323/2323 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step
Mean Squared Error: 2.174000867948278
Persentase Error: 8.105759212774283%


In [ ]:
# Contoh prediksi
# Jakarta = 6.1944° S, 106.8229° E
contoh_input = pd.DataFrame({'latitude': [6.1944], 'longitude': [106.8229], 'bulan': [7]})
contoh_input_scaled = scaler.transform(contoh_input)
prediksi_suhu = model.predict(contoh_input_scaled)
print(f"Prediksi Suhu Rata-rata: {prediksi_suhu[0]}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
Prediksi Suhu Rata-rata: [28.157804]


# Klasifikasi dengan Neural Network

In [ ]:
X = df_cuaca[['latitude', 'longitude', 'bulan']]
y = df_cuaca['kategori_hujan']

y_encoded = to_categorical(y)

X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2)

# Membuat model neural network
model = Sequential()
model.add(Dense(64, activation='relu', input_dim=X_train.shape[1]))
model.add(Dense(32, activation='relu'))
model.add(Dense(y_train.shape[1], activation='softmax'))

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense_3 (Dense)                      │ (None, 64)                  │             256 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_4 (Dense)                      │ (None, 32)                  │           2,080 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_5 (Dense)                      │ (None, 3)                   │              99 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 2,435 (9.51 KB)

 Trainable params: 2,435 (9.51 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model.fit(X_train, y_train, epochs=25, batch_size=128, validation_split=0.2)

Epoch 1/25
1859/1859 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step - accuracy: 0.4196 - loss: 1.2165 - val_accuracy: 0.4681 - val_loss: 1.0376
Epoch 2/25
1859/1859 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.4533 - loss: 1.0450 - val_accuracy: 0.4672 - val_loss: 1.0352
Epoch 3/25
1859/1859 ━━━━━━━━━━━━━━━━━━━━ 12s 5ms/step - accuracy: 0.4567 - loss: 1.0403 - val_accuracy: 0.4500 - val_loss: 1.0339
Epoch 4/25
1859/1859 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step - accuracy: 0.4622 - loss: 1.0350 - val_accuracy: 0.4722 - val_loss: 1.0288
Epoch 5/25
1859/1859 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.4657 - loss: 1.0301 - val_accuracy: 0.4763 - val_loss: 1.0207
Epoch 6/25
1859/1859 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step - accuracy: 0.4686 - loss: 1.0281 - val_accuracy: 0.4732 - val_loss: 1.0222
Epoch 7/25
1859/1859 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.4703 - loss: 1.0248 - val_accuracy: 0.4717 - val_loss: 1.0276
Epoch 8/25
1859/1859 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.4731 - loss: 1.0243 -

In [ ]:
# menyimpan model
model.save('model_klasifikasi.h5')

In [ ]:
# Contoh prediksi
# Jakarta = 6.1944° S, 106.8229° E
contoh_input = pd.DataFrame({'latitude': [6.1944], 'longitude': [106.8229], 'bulan': [12]})
prediksi_hujan = model.predict(contoh_input)
kategori_prediksi = np.argmax(prediksi_hujan)
print(f"Prediksi Kategori Hujan: {kode_kategori_hujan[kategori_prediksi]}")

print("Kemungkinan (tidak hujan, hujan ringan, hujan lebat)", prediksi_hujan)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
Prediksi Kategori Hujan: Hujan Lebat
Kemungkinan (tidak hujan, hujan ringan, hujan lebat) [[0.3425755 0.3076497 0.3497748]]
